# Installing Dependencies


In [2]:
# use this cell to install packages if needed
#!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu116 --upgrade

In [3]:
# use this cell to install packages if needed
#!pip install transformers --upgrade

In [4]:
#!pip install tqdm

In [5]:
#!pip install tensorboard

# Setup

In [ ]:
import json
import os
import timeit
import collections
import time
from pprint import pprint
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, squad_convert_examples_to_features
from transformers.data.processors.squad import SquadV2Processor,SquadResult
from transformers.data.metrics.squad_metrics import (
    compute_predictions_log_probs,
    compute_predictions_logits,
    squad_evaluate,
)

2026-08-20 19:26:45.095771: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
DO_LOWER_CASE = True
NBEST_SIZE = 20
DOC_STRIDE = 128
MAX_SEQ_LENGTH = 384
MAX_QUERY_LENGTH = 64
MAX_ANSWER_LENGTH = 30

# مسیر پوشه داده‌ها روی محیط ابری شما
DATA_DIR = './Data/Squad'
PREDICT_FILE = 'dev-v2.0.json'

BERT_MODEL_TYPE = 'bert'
BERT_MODEL_HF_PATH = "twmkn9/bert-base-uncased-squad2"
BERT_OUTPUT_DIR = "models/bert/twmkn9_bert-base-uncased-squad2"

DISTILBERT_MODEL_TYPE = 'distilbert'
DISTILBERT_MODEL_HF_PATH = 'twmkn9/distilbert-base-uncased-squad2'
DISTILBERT_OUTPUT_DIR = 'models/distilbert/twmkn9_distilbert-base-uncased-squad2'

print("Configurations are set successfully!")

# Q&A Challenges

Measuring the success of Q&A systems is complicated. When asked a question like "Why the sky is Blue?", there are several potential right answers. For instance, one could refer to "Rayleigh Scattering" or another answer could be:
```
The Earth's atmosphere scatters short-wavelength light more efficiently than that of longer wavelengths. Because its wavelengths are shorter, blue light is more strongly scattered than the longer-wavelength lights, red or green. Hence the result that when looking at the sky away from the direct incident sunlight, the human eye perceives the sky to be blue.
```

Both options are correct and referred in Wikipedia Artile 'Diffuse Sky Radiation'


Most Q&A systems rely on a corpus of information that is initially indexed by an information retrieval system. Then, snippets of text are extracted where the Q&A model scores the most likely sentences to answer a given query.

However, the same snippet of text that answers the blue sky question, may not be able to answer a similar query like "Could the Sky ever be green?"

This is the gray area for measuring performance of Q&A models. How should we judge a model's success when there are multiple correct answers, even more incorrect answers, and, potentially no answers available to it at all?

# SQuAD Dataset

```
Stanford Question Answering Dataset (SQuAD) is a reading comprehension dataset, consisting of questions posed by crowdworkers on a set of Wikipedia articles, where the answer to every question is a segment of text, or span, from the corresponding reading passage, or the question might be unanswerable.
```

```
SQuAD2.0 combines the 100,000 questions in SQuAD1.1 with over 50,000 unanswerable questions written adversarially by crowdworkers to look similar to answerable ones. To do well on SQuAD2.0, systems must not only answer questions when possible, but also determine when no answer is supported by the paragraph and abstain from answering.
```

We target the SQuAD 2.0 as it represents a scenario that is closer to the real world: **It includes additional questions that cannot be answered by the accompanying passage**

**Download the Squad dev set for model evaluation**

In [ ]:
#!wget -P data/squad/ https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json

We will make extensive use of [Hugging Face](https://huggingface.co/) and [Pytorch](https://pytorch.org/) throughout this code, they provide several implementations for loading datasets and models

## Loading the DEV set using Hugging Face data processors

We will make use of [Processors](https://huggingface.co/transformers/main_classes/processors.html) to facilitate basic processing tasks with some canonical NLP datasets. The processors can be used for loading datasets and converting their examples to features for direct use in the model. More specifically, we will be using the [SQuAD processors](https://huggingface.co/transformers/main_classes/processors.html#squad)

In [ ]:
def to_list(tensor):
    return tensor.detach().cpu().tolist()

In [ ]:
def load_and_cache_examples(model_name_or_path, 
                            data_dir= DATA_DIR, 
                            predict_file=PREDICT_FILE, 
                            max_seq_length=MAX_SEQ_LENGTH, 
                            doc_stride=DOC_STRIDE, 
                            max_query_length=MAX_QUERY_LENGTH, 
                            overwrite_cache=True):
    
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=False)
    # Load data features from cache or dataset file
    input_dir = data_dir if data_dir else "."
    cached_features_file = os.path.join(
        input_dir,
        "cached_{}_{}_{}".format(
            "dev",
            list(filter(None, model_name_or_path.split("/"))).pop(),
            str(max_seq_length),
        ),
    )

    # Init features and dataset from cache if it exists
    if os.path.exists(cached_features_file) and not overwrite_cache:
        logger.info("Loading features from cached file %s", cached_features_file)
        features_and_dataset = torch.load(cached_features_file)
        features, dataset, examples = (
            features_and_dataset["features"],
            features_and_dataset["dataset"],
            features_and_dataset["examples"],
        )
    else:

        processor = SquadV2Processor()

        examples = processor.get_dev_examples(data_dir, filename=predict_file)

        features, dataset = squad_convert_examples_to_features(
            examples=examples,
            tokenizer=tokenizer,
            max_seq_length=max_seq_length,
            doc_stride=doc_stride,
            max_query_length=max_query_length,
            is_training=False,
            return_dataset="pt",
            threads=1,
        )


    return dataset, examples, features

In [11]:
dataset, examples, features = load_and_cache_examples(BERT_MODEL_HF_PATH)

In [12]:
print(f'There are {len(examples)} in the dev dataset')

There are 11873 in the dev dataset


The list of examples contains objects of type **transformers.data.processors.squad.SquadExample**. We use the function below to extract the information we want from such objects. More specifically: **'qid'**, **'question_text'**, **'context_text'** and **'answer'**

We will firstly create some extra variables to help on manipulation of data

In [13]:
# generate some maps to help us identify examples of interest
qid_to_example_index = {example.qas_id: i for i, example in enumerate(examples)}
qid_to_has_answer = {example.qas_id: bool(example.answers) for example in examples}
answer_qids = [qas_id for qas_id, has_answer in qid_to_has_answer.items() if has_answer]
no_answer_qids = [qas_id for qas_id, has_answer in qid_to_has_answer.items() if not has_answer]

And also, the function below to help on extracting information given a `qid` (question unique identifier)

In [14]:
def display_example(qid:str):    
    from pprint import pprint

    idx = qid_to_example_index[qid]
    q = examples[idx].question_text
    c = examples[idx].context_text
    a = [answer['text'] for answer in examples[idx].answers]
    
    print(f'Example {idx} of {len(examples)}\n---------------------')
    print(f"Q: {q}\n")
    print("Context:")
    pprint(c)
    print(f"\nTrue Answers:\n{a}")